# 100 · Fusion + Dashboard

Combine domain scores, compute fusion metrics, and preview dashboard components.

In [1]:
# Helper: clean DataFrame to avoid missing-column/NaN errors
import pandas as pd

def clean_frame(df, target=None, numeric_expected=None, categorical_expected=None):
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    df = df.dropna(axis=1, how='all').copy()
    numeric_expected = numeric_expected or []
    categorical_expected = categorical_expected or []
    for col in numeric_expected:
        if col not in df.columns:
            df[col] = 0.0
    for col in categorical_expected:
        if col not in df.columns:
            df[col] = 'missing'
    num_cols = df.select_dtypes(include=['number']).columns
    cat_cols = df.select_dtypes(exclude=['number']).columns
    if len(num_cols):
        df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    if len(cat_cols):
        df[cat_cols] = df[cat_cols].fillna('missing')
    if target:
        if target not in df.columns:
            raise KeyError(f"Target '{target}' missing. Columns: {df.columns.tolist()}")
        df[target] = pd.to_numeric(df[target], errors='coerce').fillna(0).astype(int)
    return df


In [3]:
from pathlib import Path
import sys

# Navigate to project root (notebooks/overview -> project root)
project_root = Path('../..').resolve()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from uais.fusion.train_fusion_meta import build_fusion_dataset, train_fusion_model

print('Project root:', project_root)
print('Fusion module imported successfully')

Project root: /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2
Fusion module imported successfully


In [5]:
# Demo: Fusion model combines scores from multiple domains
# This demonstrates the architecture - in production, domain models produce scores that feed into fusion

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

# Simulate domain scores (in production these come from fraud, cyber, behavior models)
np.random.seed(42)
n_samples = 1000

# Generate synthetic multi-domain scores
fraud_scores = np.random.beta(2, 5, n_samples)  # Fraud model outputs
cyber_scores = np.random.beta(2, 5, n_samples)  # Cyber model outputs  
behavior_scores = np.random.beta(2, 5, n_samples)  # Behavior model outputs

# Fusion features: stack domain scores
X_fusion = np.column_stack([fraud_scores, cyber_scores, behavior_scores])

# Ground truth labels (simulated: anomaly if any domain score is high)
y_true = ((fraud_scores > 0.6) | (cyber_scores > 0.6) | (behavior_scores > 0.6)).astype(int)

print(f"Fusion dataset shape: {X_fusion.shape}")
print(f"Label distribution: {np.bincount(y_true)}")

# Train fusion meta-model
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_fusion, y_true, test_size=0.2, random_state=42, stratify=y_true)

fusion_model = LogisticRegression(max_iter=200, class_weight='balanced')
fusion_model.fit(X_train, y_train)

# Evaluate
y_proba = fusion_model.predict_proba(X_test)[:, 1]
y_pred = (y_proba > 0.5).astype(int)

print(f"\nFusion Model ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Anomaly']))

Fusion dataset shape: (1000, 3)
Label distribution: [895 105]

Fusion Model ROC-AUC: 0.8712

Classification Report:
              precision    recall  f1-score   support

      Normal       0.97      0.79      0.87       179
     Anomaly       0.30      0.76      0.43        21

    accuracy                           0.79       200
   macro avg       0.63      0.78      0.65       200
weighted avg       0.90      0.79      0.83       200



/Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept


## Update reports/tables after fusion
Regenerate dashboard/report artifacts from the latest fusion run.

In [6]:
# Generate consolidated report + per-domain tables
!python ../src/scripts/generate_reports.py
!python ../src/uais/reporting/make_tables.py

python: can't open file '/Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/notebooks/overview/../src/scripts/generate_reports.py': [Errno 2] No such file or directory
python: can't open file '/Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/notebooks/overview/../src/uais/reporting/make_tables.py': [Errno 2] No such file or directory
python: can't open file '/Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/notebooks/overview/../src/uais/reporting/make_tables.py': [Errno 2] No such file or directory


In [7]:
import json
import pandas as pd
summary_path = project_root / 'experiments' / 'report_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    display(pd.json_normalize(summary, sep='->'))
fusion_metrics_csv = project_root / 'experiments' / 'fusion' / 'metrics' / 'metrics.csv'
if fusion_metrics_csv.exists():
    display(pd.read_csv(fusion_metrics_csv))


,cyber->metrics->roc_auc,cyber->metrics->pr_auc,cyber->metrics->f1,cyber->metrics->precision,cyber->metrics->recall,cyber->metrics->accuracy,cyber->metrics->cv_roc_auc_mean,cyber->metrics->cv_scores,cyber->metrics->roc_auc_ci_lower,cyber->metrics->roc_auc_ci_upper,cyber->runtime->train_time_sec,cyber->runtime->predict_proba_sec_per_run
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,"[1.0, 1.0, 1.0]",0.9999999999999999,1.0,3.727215,0.07017


,Metric,Value
0,roc_auc,0.886983
1,f1,0.451613
2,cv_roc_auc_mean,0.848943


In [9]:
# Dashboard preview - run in terminal with:
# streamlit run app/streamlit_chatbot/app.py

print("✅ Fusion model demonstrated successfully!")
print("\nTo launch full dashboard, run:")
print("  SENTINELFORGE_BACKEND=http://localhost:8000 streamlit run app/streamlit_chatbot/app.py")
print("\nFusion weights (domain importance):")
print(f"  Fraud weight:    {fusion_model.coef_[0][0]:.4f}")
print(f"  Cyber weight:    {fusion_model.coef_[0][1]:.4f}")
print(f"  Behavior weight: {fusion_model.coef_[0][2]:.4f}")

✅ Fusion model demonstrated successfully!

To launch full dashboard, run:
  SENTINELFORGE_BACKEND=http://localhost:8000 streamlit run app/streamlit_chatbot/app.py

Fusion weights (domain importance):
  Fraud weight:    4.7876
  Cyber weight:    5.6997
  Behavior weight: 4.3398
